In [2]:
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

model_name = "/gpfs/projects/bsc14/MN4/bsc14/models/entity_linking/general/sapbert_15_parents_1epoch"

st_model = SentenceTransformer(model_name)


/gpfs/projects/bsc14/code/nlp4bia/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
No sentence-transformers model found with name /gpfs/projects/bsc14/MN4/bsc14/models/entity_linking/general/sapbert_15_parents_1epoch. Creating a new one with mean pooling.


In [3]:
from nlp4bia.datasets.benchmark.medprocner import MedprocnerLoader, MedprocnerGazetteer

df_proc = MedprocnerLoader().df
gaz_proc = MedprocnerGazetteer().df
print(df_proc.shape, gaz_proc.shape)

preprocessing data...
preprocessing data...
(8475, 11) (234674, 5)


In [4]:
gaz_proc = gaz_proc.sort_values(by=["code", "mainterm"], ascending=[True, False])
gaz_proc

,code,language,term,semantic_tag,mainterm
28746,10002003,es,resección del fondo gástrico,procedure,1
30180,10002003,es,resección de la cúpula del estómago,procedure,0
145401,10002003,es,fundectomía gástrica,procedure,0
32123,10006000,es,reparación de un desprendimiento de la retina ...,procedure,1
35910,10006000,es,reinserción de la retina por fotocoagulación c...,procedure,0
...,...,...,...,...,...
22489,9990009,es,servicio de planificación del tratamiento de t...,procedure,0
41385,9992001,es,radioisótopo de molibdeno,substance,1
137445,9993006,es,incisión del diafragma,procedure,1
204227,9996003,es,artrotomía con drenaje de la articulación tars...,procedure,1


In [5]:
vector_db = st_model.encode(gaz_proc["term"].tolist()[:100], show_progress_bar=True, convert_to_tensor=True, normalize_embeddings=True)

Batches: 100%|██████████| 4/4 [00:00<00:00,  5.43it/s]


In [6]:
from nlp4bia.linking.retrievers import DenseRetriever

In [10]:
biencoder = DenseRetriever(vector_db=vector_db, model=st_model, )
biencoder.retrieve_top_k(["reparación de un desprendimiento de la retina"], gaz_proc.iloc[:100], k=10, input_format="text")

Batches: 100%|██████████| 1/1 [00:00<00:00, 179.92it/s]

Getting the indices...
I shape: (1, 100)
k 10
Getting the distances...
Getting the terms and codes ...


[{'codes': ['10006000',
   '10006000',
   '10006000',
   '10012005',
   '1002223009',
   '1003626004',
   '10029008',
   '10012005',
   '10002003',
   '1003625000'],
  'terms': ['reparación de un desprendimiento de la retina por fotocoagulación con arco xenón',
   'fotocoagulación con arco xenón para reparación de desprendimiento de retina',
   'reinserción de la retina por fotocoagulación con arco xenón',
   'extracción mediante presión',
   'evaluación del progreso hacia el logro de los objetivos para lograr la seguridad alimentaria',
   'plástico para escribir para personas con discapacidad visual',
   'precauciones para evitar un suicidio',
   'expresión',
   'fundectomía gástrica',
   'papel para escribir para personas con discapacidad visual'],
  'similarity': [0.8431774377822876,
   0.7645474672317505,
   0.7163451910018921,
   0.4380110204219818,
   0.40896904468536377,
   0.4088270962238312,
   0.4078870713710785,
   0.4045633375644684,
   0.4044119715690613,
   0.393970280885